# Import Required Libraries


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read Parquet Files from Microsoft Fabric Lakehouse

In [3]:
path = "abfss://Project_Customer360@onelake.dfs.fabric.microsoft.com/Customer360_LH.Lakehouse/Files/bronze/"


customers_raw = spark.read.parquet(path + "customers.parquet")
orders_raw = spark.read.parquet(path + "orders.parquet")
payments_raw = spark.read.parquet(path + "payments.parquet")
support_raw = spark.read.parquet(path + "support_tickets.parquet")
web_raw = spark.read.parquet(path + "web_activities.parquet")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 5, Finished, Available, Finished, False)

# Store Raw Data as Delta Tables

In [4]:
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")

orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")

payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")

support_raw.write.format("delta").mode("overwrite").saveAsTable("support_tickets")

web_raw.write.format("delta").mode("overwrite").saveAsTable("web_activities")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 6, Finished, Available, Finished, False)

# Customer Data Cleaning

In [7]:
customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn(
        "gender",
        when(lower(col("gender")).isin("f", "female"), "Female")
        .when(lower(col("gender")).isin("m", "male"), "Male")
        .otherwise("Other")
    )
    .withColumn(
        "dob",
        to_date(regexp_replace(col("dob"), "/", "-"))
    )
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)

customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 9, Finished, Available, Finished, False)

# Orders Data Cleaning

In [8]:
orders_clean = (
    orders_raw
    .withColumn(
        "order_date",
        when(col("order_date").rlike("^\\d{4}/\\d{2}/\\d{2}$"),
             to_date(col("order_date"), "yyyy/MM/dd"))
        .when(col("order_date").rlike("^\\d{2}-\\d{2}-\\d{4}$"),
              to_date(col("order_date"), "dd-MM-yyyy"))
        .when(col("order_date").rlike("^\\d{8}$"),
              to_date(col("order_date"), "yyyyMMdd"))
        .otherwise(
            to_date(col("order_date"), "yyyy-MM-dd")
        )
    )
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn(
        "amount",
        when(col("amount") < 0, None).otherwise(col("amount"))
    )
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)

orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 10, Finished, Available, Finished, False)

# Payments Data Cleaning

In [9]:
payments_clean = (
    payments_raw
    .withColumn(
        "payment_date",
        to_date(regexp_replace(col("payment_date"), "/", "-"))
    )
    .withColumn(
        "payment_method",
        initcap(col("payment_method"))
    )
    .replace(
        {"creditcard": "Credit Card"},
        subset=["payment_method"]
    )
    .withColumn(
        "payment_status",
        initcap(col("payment_status"))
    )
    .withColumn(
        "amount",
        col("amount").cast(DoubleType())
    )
    .withColumn(
        "amount",
        when(col("amount") < 0, None).otherwise(col("amount"))
    )
    .dropna(
        subset=["customer_id", "payment_date", "amount"]
    )
)

payments_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_payments")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 11, Finished, Available, Finished, False)

# Support Tickets Data Cleaning

In [10]:
support_clean = (
    support_raw
    .withColumn(
        "ticket_date",
        to_date(regexp_replace(col("ticket_date"), "/", "-"))
    )
    .withColumn(
        "issue_type",
        initcap(trim(col("issue_type")))
    )
    .withColumn(
        "resolution_status",
        initcap(trim(col("resolution_status")))
    )
    .replace(
        {"NA": None, "": None},
        subset=["issue_type", "resolution_status"]
    )
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)

support_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_support")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 12, Finished, Available, Finished, False)

# Web Activity Data Cleaning

In [11]:
web_clean = (
    web_raw
    .withColumn(
        "session_time",
        to_date(regexp_replace(col("session_time"), "/", "-"))
    )
    .withColumn(
        "page_viewed",
        lower(col("page_viewed"))
    )
    .withColumn(
        "device_type",
        initcap(col("device_type"))
    )
    .dropDuplicates(["session_id"])
    .dropna(
        subset=["customer_id", "session_time", "page_viewed"]
    )
)

web_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_web")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 13, Finished, Available, Finished, False)

# Load Silver Tables

In [12]:
cust = spark.table("silver_customers").alias("c")

orders = spark.table("silver_orders").alias("o")

payments = spark.table("silver_payments").alias("p")

support = spark.table("silver_support").alias("s")

web = spark.table("silver_web").alias("w")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 14, Finished, Available, Finished, False)

# Build Customer 360 Gold Layer

In [13]:
customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)

customer360.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer360")

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 15, Finished, Available, Finished, False)

# Preview Final Dataset

In [14]:
display(customer360.limit(5))

StatementMeta(, a932e1ad-9b88-4db6-9fe1-37a7d7ae8c39, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ba14363-4097-44eb-ae93-27ca3ed3f322)